In [23]:
from flax import nnx
import jax
import jax.numpy as jnp


class QNetwork(nnx.Module):
    def __init__(self, rngs: nnx.Rngs, d_in=8, d_hidden_1=128, d_hidden_2=128, d_out=4):
        self.lin_in = nnx.Linear(d_in, d_hidden_1, rngs=rngs)
        self.hidden_1 = nnx.Linear(d_hidden_1, d_hidden_2, rngs=rngs)
        self.hidden_2 = nnx.Linear(d_hidden_2, d_hidden_1, rngs=rngs)
        self.lin_out = nnx.Linear(d_hidden_1, d_out, rngs=rngs)


    def __call__(self, x):
        x = nnx.relu(self.lin_in(x))
        x = nnx.relu(self.hidden_1(x))
        x = nnx.relu(self.hidden_2(x))
        return self.lin_out(x)


In [24]:



def act_epsilon_greedy(key, q_values, epsilon=0.05):
    prob_key, action_key = jax.random.split(key)
    random_val = jax.random.uniform(prob_key)
    random_action = jax.random.randint(action_key, shape=(), minval=0, maxval=4)
    greedy_action = jnp.argmax(q_values)
    return jnp.where(random_val < epsilon, random_action, greedy_action)

In [25]:
BATCH_SIZE = 16
rngs = nnx.Rngs(0)
model = QNetwork(rngs)

key = jax.random.key(0)
dummy_q_values = jax.random.uniform(key, (BATCH_SIZE, 4))


In [26]:

# Prediction
## Get a batch of states -> From buffer

dummy_states = jax.random.uniform(key, (16, 8))
dummy_q_values = model(dummy_states) # batch inference on the model


fun_batch_act = nnx.vmap(act_epsilon_greedy, in_axes=(0, 0, None)) # key and state need to have 0
batch_keys = jax.random.split(key, BATCH_SIZE)
chosen_actions = fun_batch_act(batch_keys, dummy_q_values, 0.5)

# i now have all q values, states and chosen actions

# I need, gradient, based on the loss function


In [27]:
dummy_next_states = jax.random.uniform(key, (BATCH_SIZE, 8))
dummy_rewards = jax.random.randint(key, (BATCH_SIZE, ), 5, 10)
dummy_is_terminal = jax.random.randint(key, (BATCH_SIZE, ), 0, 2)

In [28]:
def calcualte_y(eval_model, next_state, reward, is_terminal_state, gamma=0.9):
    return jnp.where(is_terminal_state,
                     reward,
                     reward + gamma * jnp.max(eval_model(next_state))
                    )

batched_calculate_target = nnx.vmap(calcualte_y, in_axes=(None, 0,0,0, None))
batched_targets = batched_calculate_target(model, dummy_next_states, dummy_rewards, dummy_is_terminal, 0.9)


In [29]:
def mse_loss(model, states, actions, targets):
    qs = model(states)
    # Pick the Q-value for the action taken in each state
    # This selects qs[0, actions[0]], qs[1, actions[1]], etc.
    chosen_qs = qs[jnp.arange(len(actions)), actions]
    loss = jnp.mean((chosen_qs - targets) ** 2)
    return loss, qs

@jax.jit
def train_step(model, optimizer, states, chosen_actions, targets):
    fun_inference_grad = nnx.value_and_grad(mse_loss, has_aux=True)
    (loss, qs), grad = fun_inference_grad(model, states, chosen_actions, targets)
    optimizer.update(model, grad)
    return loss, qs

In [ ]:
import optax

LR = 0.001
MOMENTUM = 0.9

optimizer = nnx.Optimizer(
  model, optax.adam(LR), wrt=nnx.Param
)

loss, qs = train_step(model, optimizer, dummy_states, chosen_actions, batched_targets)

54.071487
